# Gradient Boosting Survival Model (R)

Component-wise gradient boosting for Cox proportional hazards using `CoxBoost`.
Cross-validation selects the optimal number of boosting steps; a penalty grid
searches for the best regularisation strength.

| Python (scikit-survival) | R (CoxBoost) |
|--------------------------|--------------|
| `GradientBoostingSurvivalAnalysis` | `CoxBoost::CoxBoost()` |
| `RandomizedSearchCV` (n_estimators, lr, ...) | `cv.CoxBoost()` + penalty grid |
| `permutation_importance` | Coefficient magnitudes |

**Libraries:** `CoxBoost`, `survival`, `survminer`, `ggplot2`, `reshape2`, `pheatmap`

In [ ]:
# install.packages(c('CoxBoost','survival','survminer','ggplot2','dplyr','reshape2','pheatmap','ggpubr'))

In [ ]:
suppressPackageStartupMessages({
  library(survival)
  library(CoxBoost)
  library(survminer)
  library(ggplot2)
  library(dplyr)
  library(reshape2)
  library(pheatmap)
  library(ggpubr)
})

DATA_DIR  <- '../../datasets/csv_files'
VIS_DIR   <- '../../visuals'
MODEL_DIR <- '../../models'
dir.create(VIS_DIR,   showWarnings=FALSE, recursive=TRUE)
dir.create(MODEL_DIR, showWarnings=FALSE, recursive=TRUE)
set.seed(42)
message('Packages loaded.')

## Preparing Data

In [ ]:
load_surv <- function(path) {
  df <- read.csv(path, stringsAsFactors=FALSE)
  df[complete.cases(df[c('relapse_free_time','relapse_free_event')]), ]
}

train_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/train_data.csv'))
test1_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/test_data_one.csv'))
test2_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/test_data_two.csv'))
test3_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/test_data_three.csv'))

surv_meta <- c('sample_name','relapse_free_time','relapse_free_event')
gene_cols <- setdiff(names(train_raw), surv_meta)
train_raw$relapse_free_event <- as.integer(train_raw$relapse_free_event)

make_matrix <- function(df, genes) {
  as.matrix(df[, intersect(genes, names(df)), drop=FALSE])
}

time_train   <- train_raw$relapse_free_time
status_train <- as.integer(train_raw$relapse_free_event)
X_train      <- make_matrix(train_raw, gene_cols)

time_t1  <- test1_raw$relapse_free_time;  status_t1 <- as.integer(test1_raw$relapse_free_event)
time_t2  <- test2_raw$relapse_free_time;  status_t2 <- as.integer(test2_raw$relapse_free_event)
time_t3  <- test3_raw$relapse_free_time;  status_t3 <- as.integer(test3_raw$relapse_free_event)
X_t1     <- make_matrix(test1_raw, gene_cols)
X_t2     <- make_matrix(test2_raw, gene_cols)
X_t3     <- make_matrix(test3_raw, gene_cols)

cat(sprintf('Train: %d | Test1: %d | Test2: %d | Test3: %d | Genes: %d\n',
            nrow(X_train), nrow(X_t1), nrow(X_t2), nrow(X_t3), ncol(X_train)))

## Initial CoxBoost Fit

Default penalty = 9 × events/n (Binder & Schumacher 2008).

In [ ]:
default_penalty <- 9 * sum(status_train) / length(time_train)
cat(sprintf('Default penalty: %.2f\n', default_penalty))

cat('Fitting initial CoxBoost (100 steps)...\n')
cb_initial <- CoxBoost(time=time_train, status=status_train, x=X_train,
                       stepno=100, penalty=default_penalty)

lp_init <- as.vector(X_train %*% cb_initial$coefficients[cb_initial$stepno, ])
ci_init <- concordance(Surv(time_train, status_train) ~ lp_init)$concordance
cat(sprintf('Initial train C-index: %.4f\n', ci_init))

## Cross-Validation for Optimal Stepno

Analogous to tuning `n_estimators` in scikit-survival's `GradientBoostingSurvivalAnalysis`.

In [ ]:
cat('Cross-validating for optimal stepno (K=5, maxstepno=500)...\n')
cv_result <- cv.CoxBoost(time=time_train, status=status_train, x=X_train,
                         maxstepno=500, K=5, type='verweij',
                         penalty=default_penalty)
optimal_stepno <- cv_result$optimal.step
cat(sprintf('Optimal stepno: %d\n', optimal_stepno))

## Penalty Grid Search

Analogous to tuning `learning_rate` in the Python notebook — higher penalty
= smaller effective step size = more regularisation.

In [ ]:
penalty_grid <- default_penalty * c(0.1, 0.5, 1.0, 2.0, 5.0, 10.0)
cv_scores    <- numeric(length(penalty_grid))

for (i in seq_along(penalty_grid)) {
  cv_i <- cv.CoxBoost(time=time_train, status=status_train, x=X_train,
                      maxstepno=optimal_stepno, K=5, type='verweij',
                      penalty=penalty_grid[i])
  cv_scores[i] <- max(cv_i$cv.res)
  cat(sprintf('  penalty=%.2f  CV score=%.4f  stepno=%d\n',
              penalty_grid[i], cv_scores[i], cv_i$optimal.step))
}

best_idx     <- which.max(cv_scores)
best_penalty <- penalty_grid[best_idx]
cat(sprintf('\nBest penalty: %.2f\n', best_penalty))

cv_best <- cv.CoxBoost(time=time_train, status=status_train, x=X_train,
                       maxstepno=500, K=5, type='verweij', penalty=best_penalty)
best_stepno <- cv_best$optimal.step
cat(sprintf('Best stepno at best penalty: %d\n', best_stepno))

## Final Model

In [ ]:
cat('Fitting final CoxBoost model...\n')
cb_final <- CoxBoost(time=time_train, status=status_train, x=X_train,
                     stepno=best_stepno, penalty=best_penalty)

final_coefs <- cb_final$coefficients[cb_final$stepno, ]

lp_train <- as.vector(X_train %*% final_coefs)
lp_t1    <- as.vector(X_t1 %*% final_coefs[colnames(X_t1)])
lp_t2    <- as.vector(X_t2 %*% final_coefs[colnames(X_t2)])
lp_t3    <- as.vector(X_t3 %*% final_coefs[colnames(X_t3)])

ci_train <- concordance(Surv(time_train, status_train) ~ lp_train)$concordance
ci_t1    <- concordance(Surv(time_t1,    status_t1)    ~ lp_t1)$concordance
ci_t2    <- concordance(Surv(time_t2,    status_t2)    ~ lp_t2)$concordance
ci_t3    <- concordance(Surv(time_t3,    status_t3)    ~ lp_t3)$concordance

cat(sprintf('Train C-index: %.4f\nTest 1: %.4f\nTest 2: %.4f\nTest 3: %.4f\n',
            ci_train, ci_t1, ci_t2, ci_t3))

## Variable Importance (Final Coefficient Magnitudes)

In [ ]:
vimp_df <- data.frame(
  gene        = colnames(X_train),
  coefficient = as.vector(final_coefs),
  abs_coef    = abs(as.vector(final_coefs)),
  stringsAsFactors = FALSE
)
vimp_df <- vimp_df[order(vimp_df$abs_coef, decreasing=TRUE), ]
cat('Top 15 genes by |coefficient|:\n')
print(head(vimp_df, 15))

sig_genes <- vimp_df$gene[vimp_df$abs_coef > 0][seq_len(min(8, sum(vimp_df$abs_coef > 0)))]
cat(sprintf('\nGB Gene Signature (%d genes): %s\n', length(sig_genes),
            paste(sig_genes, collapse=', ')))

PAPER_GENES <- c('TSLP','BIRC5','S100B','MDK','S100P','RARRES3','BLNK','ACO1')
overlap <- intersect(sig_genes, PAPER_GENES)
cat(sprintf('Paper overlap: %d/%d — %s\n', length(overlap),
            length(PAPER_GENES), paste(overlap, collapse=', ')))

In [ ]:
nonzero_vimp <- head(vimp_df[vimp_df$abs_coef > 0, ], 20)

p_vimp <- ggplot(nonzero_vimp, aes(x=coefficient, y=reorder(gene, coefficient))) +
  geom_col(aes(fill=coefficient > 0), alpha=0.8) +
  scale_fill_manual(values=c('TRUE'='#E64B35','FALSE'='#4DBBD5'),
                    labels=c('TRUE'='Danger (HR>1)','FALSE'='Protective (HR<1)'),
                    name='') +
  geom_vline(xintercept=0, colour='grey40') +
  labs(title='CoxBoost — Variable Importance (Top 20 non-zero genes)',
       x='Final coefficient', y=NULL) +
  theme_bw(base_size=10) +
  theme(plot.title=element_text(face='bold'), legend.position='bottom')

p_vimp
ggsave(file.path(VIS_DIR, 'gb_variable_importance.png'), p_vimp, dpi=150, width=8, height=6)
cat('Saved: gb_variable_importance.png\n')

In [ ]:
# Coefficient path over boosting steps
coef_path <- as.data.frame(cb_final$coefficients)
names(coef_path) <- colnames(X_train)
coef_path$step <- seq_len(nrow(coef_path))

top_path_genes <- head(vimp_df$gene[vimp_df$abs_coef > 0], 10)
path_long <- melt(coef_path[, c('step', top_path_genes)],
                  id.vars='step', variable.name='gene', value.name='coef')

ggplot(path_long, aes(x=step, y=coef, colour=gene, group=gene)) +
  geom_line(linewidth=0.8) +
  geom_hline(yintercept=0, linetype='dashed', colour='grey40') +
  geom_vline(xintercept=best_stepno, colour='black', linetype='dotted') +
  labs(title='CoxBoost — Coefficient Path (top selected genes)',
       x='Boosting step', y='Coefficient', colour='Gene') +
  theme_bw(base_size=10) +
  theme(plot.title=element_text(face='bold'))

## Patient Stratification & Kaplan–Meier Curves

In [ ]:
train_median_risk <- median(lp_train)
risk_label <- function(lp, cutoff) ifelse(lp >= cutoff, 'High Risk', 'Low Risk')

make_km_df <- function(lp, time, event) {
  data.frame(time=time, event=as.integer(event),
             risk_group=factor(risk_label(lp, train_median_risk),
                               levels=c('Low Risk','High Risk')))
}

km_plot <- function(df_km, title) {
  fit <- survfit(Surv(time, event) ~ risk_group, data=df_km)
  ggsurvplot(fit, data=df_km, pval=TRUE, conf.int=TRUE,
             palette=c('Low Risk'='#4DBBD5','High Risk'='#E64B35'),
             title=title, xlab='Time (days)', ylab='Relapse-Free Survival',
             legend.labs=c('Low Risk','High Risk'),
             ggtheme=theme_bw(base_size=9))$plot +
    theme(plot.title=element_text(face='bold', size=10))
}

km_plots <- list(
  km_plot(make_km_df(lp_train, time_train, status_train), 'Train'),
  km_plot(make_km_df(lp_t1,    time_t1,    status_t1),   'Test 1'),
  km_plot(make_km_df(lp_t2,    time_t2,    status_t2),   'Test 2'),
  km_plot(make_km_df(lp_t3,    time_t3,    status_t3),   'Test 3')
)
ggarrange(plotlist=km_plots, ncol=4, common.legend=TRUE, legend='bottom')

## Save Model & Results

In [ ]:
saveRDS(cb_final, file.path(MODEL_DIR, 'coxboost_model.rds'))
write.csv(vimp_df, file.path(DATA_DIR, 'gb_importance.csv'), row.names=FALSE)

cat('=== CoxBoost Summary ===\n')
cat(sprintf('Best penalty:  %.2f\n', best_penalty))
cat(sprintf('Best stepno:   %d\n',   best_stepno))
cat(sprintf('C-index — Train: %.4f | Test1: %.4f | Test2: %.4f | Test3: %.4f\n',
            ci_train, ci_t1, ci_t2, ci_t3))
cat('Saved: coxboost_model.rds, gb_importance.csv\n')